# Pratica — Lezione 10: completa il `worker.py` reale di RAG su PDF

Questo esercizio riproduce, quasi riga per riga, la coppia di template e
soluzione più chiara di tutto il materiale di IBM: il progetto
`wbphl-build_own_chatbot_without_open_ai` ("Personal Data Assistant")
distribuisce un `worker.py` con vari vuoti irrisolti, e un
`Worker_completed.py` con la soluzione esatta di quegli stessi vuoti.

Completerai tu i 4 vuoti reali:

1. Creare il modello di embeddings.
2. Caricare il PDF con `PyPDFLoader`.
3. Spezzettare il testo con `RecursiveCharacterTextSplitter`.
4. Salvare ogni domanda e risposta in `chat_history`.

**Nota sull'LLM**: il `worker.py` originale usa `WatsonxLLM` (a pagamento).
Qui usiamo la variante 100% gratuita che lo stesso repository di IBM
include in `worker_huggingFace.py`: `HuggingFaceHub` con
`tiiuae/falcon-7b-instruct`, che richiede solo un token gratuito.

**Nota sugli embeddings**: il `worker.py` originale usa
`HuggingFaceInstructEmbeddings` (una classe di LangChain oggi in disuso).
Qui usiamo il suo equivalente moderno, `HuggingFaceEmbeddings`, con lo stesso
modello (`sentence-transformers/all-MiniLM-L6-v2`) e gli stessi argomenti —
il vuoto da completare è identico nello spirito all'originale.

In [ ]:
# Installiamo le librerie necessarie.
!pip install -q langchain langchain-community chromadb sentence-transformers pypdf reportlab huggingface_hub

In [ ]:
# Generiamo lo stesso PDF di esempio del notebook di lezione, in modo che
# questa pratica sia indipendente e non serva rieseguire quello.
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas

def crear_pdf_ejemplo(ruta="documento_ejemplo.pdf"):
    contenido = """Manuale interno di Acme Corp - Politica delle ferie e dei viaggi

    Articolo 1. Giorni di ferie.
    Tutti i dipendenti a tempo pieno di Acme Corp hanno diritto a 22
    giorni lavorativi di ferie retribuite per ogni anno completo lavorato.

    Articolo 2. Richiesta di ferie.
    Le ferie devono essere richieste con almeno 15 giorni di anticipo
    tramite il portale interno delle Risorse Umane.

    Articolo 3. Rimborso spese di viaggio.
    I dipendenti che viaggiano per motivi di lavoro possono richiedere il
    rimborso di trasporto, alloggio e pasti entro i 30 giorni
    successivi al viaggio. Il limite giornaliero per i pasti e di 40 dollari.
    """
    c = canvas.Canvas(ruta, pagesize=letter)
    texto = c.beginText(40, 750)
    texto.setFont("Helvetica", 11)
    for linea in contenido.strip().split("\n"):
        texto.textLine(linea.strip())
    c.drawText(texto)
    c.save()
    return ruta

document_path = crear_pdf_ejemplo()
print("PDF di esempio pronto in:", document_path)

In [ ]:
import os
from getpass import getpass

if "HUGGINGFACEHUB_API_TOKEN" not in os.environ:
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = getpass(
        "Incolla il tuo token gratuito di Hugging Face (https://huggingface.co/settings/tokens): "
    )

## Esercizio: completa i 4 vuoti di `worker.py`

Qui sotto trovi la stessa struttura del `worker.py` reale: una funzione
`init_llm()` che crea l'LLM e gli embeddings, una funzione
`process_document(document_path)` che indicizza il PDF e costruisce la chain di RAG,
e una funzione `process_prompt(prompt)` che risponde alle domande e salva la
cronologia.

Completa ogni `# TODO` senza guardare ancora la soluzione più sotto.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import HuggingFaceHub
from langchain.chains import RetrievalQA

DEVICE = "cpu"
embeddings_model_name = "sentence-transformers/all-MiniLM-L6-v2"

llm_hub = None
embeddings = None
conversation_retrieval_chain = None
chat_history = []


def init_llm():
    """Crea l'LLM e il modello di embeddings, una sola volta."""
    global llm_hub, embeddings

    llm_hub = HuggingFaceHub(
        repo_id="tiiuae/falcon-7b-instruct",
        model_kwargs={"temperature": 0.5, "max_new_tokens": 200},
    )

    # TODO: crea il modello di embeddings con HuggingFaceEmbeddings, usando
    # embeddings_model_name come modello e model_kwargs={"device": DEVICE}
    embeddings = None  # <-- sostituisci questo


def process_document(document_path):
    """Carica un PDF, lo spezzetta, lo indicizza in Chroma e costruisce la chain di RAG."""
    global conversation_retrieval_chain

    # TODO: carica il PDF con PyPDFLoader(document_path)
    loader = None  # <-- sostituisci questo
    documents = loader.load()

    # TODO: spezzetta il testo con RecursiveCharacterTextSplitter,
    # usando chunk_size=1024 e chunk_overlap=64
    text_splitter = None  # <-- sostituisci questo
    texts = text_splitter.split_documents(documents)

    db = Chroma.from_documents(texts, embeddings)
    retriever = db.as_retriever(search_type="mmr", search_kwargs={"k": 6, "lambda_mult": 0.25})

    conversation_retrieval_chain = RetrievalQA.from_chain_type(
        llm=llm_hub, chain_type="stuff", retriever=retriever, input_key="question"
    )


def process_prompt(prompt):
    """Fa una domanda alla chain di RAG e salva la coppia domanda-risposta."""
    global chat_history

    output = conversation_retrieval_chain({"question": prompt, "chat_history": chat_history})
    answer = output["result"]

    # TODO: aggiungi la coppia (prompt, answer) a chat_history
    # (suggerimento: chat_history e una lista; aggiungi una tupla con .append(...))

    return answer

## Testiamo la tua soluzione

Se i 4 vuoti sono stati completati correttamente, questo dovrebbe inizializzare l'LLM,
indicizzare il PDF di esempio, e rispondere correttamente a una domanda sul suo
contenuto — e `chat_history` dovrebbe avere un elemento in più dopo ogni
domanda.

In [ ]:
init_llm()
process_document(document_path)

respuesta = process_prompt("Quanti giorni di ferie ho all'anno?")
print("Risposta:", respuesta)
print("Cronologia:", chat_history)

<details>
<summary>💡 Vedi soluzione</summary>

Questo è il codice esatto della soluzione reale (`Worker_completed.py`,
adattato alla variante gratuita con Hugging Face):

```python
def init_llm():
    global llm_hub, embeddings

    llm_hub = HuggingFaceHub(
        repo_id="tiiuae/falcon-7b-instruct",
        model_kwargs={"temperature": 0.5, "max_new_tokens": 200},
    )

    # Modello di embeddings: converte ogni frammento di testo in un vettore.
    embeddings = HuggingFaceEmbeddings(
        model_name=embeddings_model_name,
        model_kwargs={"device": DEVICE},
    )


def process_document(document_path):
    global conversation_retrieval_chain

    # Carichiamo il PDF ed estraiamo il suo testo.
    loader = PyPDFLoader(document_path)
    documents = loader.load()

    # Spezzettiamo il testo in frammenti di 1024 caratteri, con 64 di sovrapposizione
    # tra frammenti consecutivi (per non tagliare un'idea a metà).
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1024, chunk_overlap=64)
    texts = text_splitter.split_documents(documents)

    db = Chroma.from_documents(texts, embeddings)
    retriever = db.as_retriever(search_type="mmr", search_kwargs={"k": 6, "lambda_mult": 0.25})

    conversation_retrieval_chain = RetrievalQA.from_chain_type(
        llm=llm_hub, chain_type="stuff", retriever=retriever, input_key="question"
    )


def process_prompt(prompt):
    global chat_history

    output = conversation_retrieval_chain({"question": prompt, "chat_history": chat_history})
    answer = output["result"]

    # Salviamo la coppia domanda-risposta per mantenere la cronologia della
    # conversazione, proprio come fa l'app reale prima di restituire la risposta.
    chat_history.append((prompt, answer))

    return answer
```
</details>

## Per continuare

- Notebook di lezione: `notebooks_clase/clase10_rag_documentos.ipynb`
- Capitolo del manuale: `manual/14_rag_documentos.md`

Sfida extra (opzionale): cambia `chunk_size=1024, chunk_overlap=64` con
`chunk_size=300, chunk_overlap=30` in `process_document`, reindicizza il
PDF e confronta le risposte. Noti qualche differenza nella precisione o nel
livello di dettaglio delle risposte?